<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 3.2.1
# *Querying the International Space Station*

## The OpenNotify API

The OpenNotify API exposes a few attributes of the International Space Station (ISS) via a simple, authentication-free interface. The simplicity of this API precludes any need for a dedicated Python library. However, as with many APIs, it accepts requests according to HTTP standards and returns responses in JSON format, so the Python libraries request and json will make managing the I/O simpler still.

In [2]:
import requests
import json
from datetime import datetime, date, time

This request fetches the latest position of the international space station:

In [3]:
response = requests.get("http://api.open-notify.org/iss-now.json")

Print the status code and text of the response:

In [4]:
print(response)

<Response [200]>


In [5]:
#ANSWER
if response.status_code == 200:
    print("Status Code:", response.status_code)
    print("Response Text:", response.text)
else:
    print("Failed to retrieve data. Status Code:", response.status_code)
    print("Response Text:", response.text)

Status Code: 200
Response Text: {"timestamp": 1770168430, "iss_position": {"latitude": "-42.2859", "longitude": "-51.3269"}, "message": "success"}


In [6]:
print(response.text)

{"timestamp": 1770168430, "iss_position": {"latitude": "-42.2859", "longitude": "-51.3269"}, "message": "success"}


In [7]:
#ANSWER


We can use another API to request the current position of the ISS and the next few times at which it will be over a certain location. The latitude and longitude of Sydney are (-33.87, 151.21).

In [8]:
response = requests.get("https://api.g7vrd.co.uk/v1/satellite-passes/25544/-33.87/151.21.json?minelevation=0&hours=24")

Print the response header:

In [9]:
#ANSWER
display(response.headers)

{'Date': 'Wed, 04 Feb 2026 01:27:11 GMT', 'Server': 'Apache', 'X-Rate-Limit-Remaining': '9', 'Vary': 'Origin,Access-Control-Request-Method,Access-Control-Request-Headers', 'Access-Control-Allow-Origin': '*', 'X-Content-Type-Options': 'nosniff', 'X-XSS-Protection': '0', 'Cache-Control': 'no-cache, no-store, max-age=0, must-revalidate', 'Pragma': 'no-cache', 'Expires': '0', 'X-Frame-Options': 'DENY', 'Content-Type': 'application/json', 'Keep-Alive': 'timeout=5, max=100', 'Connection': 'Keep-Alive', 'Transfer-Encoding': 'chunked'}

Print the content of the response (the data that the server returned):

In [10]:
#ANSWER
display(response.content)

b'{"api_status":"ALPHA","request_timestamp":"2026-02-04T01:27:11.299955155Z","norad_id":25544,"satellite_name":"ISS","tle_last_retrieved":"2026-02-03T09:06:17.108411083Z","lat":-33.87,"lon":151.21,"hours":24,"min_elevation":0,"query_ms":18,"passes":[{"start":"2026-02-04T09:01:26.281Z","tca":"2026-02-04T09:04:26.281Z","end":"2026-02-04T09:07:51.281Z","aos_azimuth":22,"los_azimuth":94,"max_elevation":4.0},{"start":"2026-02-04T10:35:16.281Z","tca":"2026-02-04T10:40:46.281Z","end":"2026-02-04T10:46:26.281Z","aos_azimuth":319,"los_azimuth":130,"max_elevation":68.0},{"start":"2026-02-04T12:13:06.281Z","tca":"2026-02-04T12:18:06.281Z","end":"2026-02-04T12:22:56.281Z","aos_azimuth":270,"los_azimuth":147,"max_elevation":15.0},{"start":"2026-02-04T13:52:56.281Z","tca":"2026-02-04T13:55:56.281Z","end":"2026-02-04T13:59:31.281Z","aos_azimuth":225,"los_azimuth":152,"max_elevation":4.0},{"start":"2026-02-04T15:31:26.281Z","tca":"2026-02-04T15:34:56.281Z","end":"2026-02-04T15:38:16.281Z","aos_azimuth

Note that this is a Python byte string:

In [11]:
print(type(response.content))

<class 'bytes'>


Print just the "content-type" value from the header:

In [12]:
#ANSWER
print(response.headers.get("content-type"))

application/json


JSON was designed to be easy for computers to read, not for people. The `requests` library can decode the JSON byte string:

In [13]:
overheads = response.json()
display(overheads)

{'api_status': 'ALPHA',
 'request_timestamp': '2026-02-04T01:27:11.299955155Z',
 'norad_id': 25544,
 'satellite_name': 'ISS',
 'tle_last_retrieved': '2026-02-03T09:06:17.108411083Z',
 'lat': -33.87,
 'lon': 151.21,
 'hours': 24,
 'min_elevation': 0,
 'query_ms': 18,
 'passes': [{'start': '2026-02-04T09:01:26.281Z',
   'tca': '2026-02-04T09:04:26.281Z',
   'end': '2026-02-04T09:07:51.281Z',
   'aos_azimuth': 22,
   'los_azimuth': 94,
   'max_elevation': 4.0},
  {'start': '2026-02-04T10:35:16.281Z',
   'tca': '2026-02-04T10:40:46.281Z',
   'end': '2026-02-04T10:46:26.281Z',
   'aos_azimuth': 319,
   'los_azimuth': 130,
   'max_elevation': 68.0},
  {'start': '2026-02-04T12:13:06.281Z',
   'tca': '2026-02-04T12:18:06.281Z',
   'end': '2026-02-04T12:22:56.281Z',
   'aos_azimuth': 270,
   'los_azimuth': 147,
   'max_elevation': 15.0},
  {'start': '2026-02-04T13:52:56.281Z',
   'tca': '2026-02-04T13:55:56.281Z',
   'end': '2026-02-04T13:59:31.281Z',
   'aos_azimuth': 225,
   'los_azimuth': 15

What kind of object did this give us?

In [14]:
#ANSWER:
print(type(overheads))

<class 'dict'>


Python dicts are easier to work with, but the data we want is still buried in that data structure, so we have to dig it out. First, extract the `passes` value to a separate list:

In [15]:
#ANSWER:
passes = overheads['passes']
display(passes)

[{'start': '2026-02-04T09:01:26.281Z',
  'tca': '2026-02-04T09:04:26.281Z',
  'end': '2026-02-04T09:07:51.281Z',
  'aos_azimuth': 22,
  'los_azimuth': 94,
  'max_elevation': 4.0},
 {'start': '2026-02-04T10:35:16.281Z',
  'tca': '2026-02-04T10:40:46.281Z',
  'end': '2026-02-04T10:46:26.281Z',
  'aos_azimuth': 319,
  'los_azimuth': 130,
  'max_elevation': 68.0},
 {'start': '2026-02-04T12:13:06.281Z',
  'tca': '2026-02-04T12:18:06.281Z',
  'end': '2026-02-04T12:22:56.281Z',
  'aos_azimuth': 270,
  'los_azimuth': 147,
  'max_elevation': 15.0},
 {'start': '2026-02-04T13:52:56.281Z',
  'tca': '2026-02-04T13:55:56.281Z',
  'end': '2026-02-04T13:59:31.281Z',
  'aos_azimuth': 225,
  'los_azimuth': 152,
  'max_elevation': 4.0},
 {'start': '2026-02-04T15:31:26.281Z',
  'tca': '2026-02-04T15:34:56.281Z',
  'end': '2026-02-04T15:38:16.281Z',
  'aos_azimuth': 205,
  'los_azimuth': 129,
  'max_elevation': 5.0},
 {'start': '2026-02-04T17:07:51.281Z',
  'tca': '2026-02-04T17:12:51.281Z',
  'end': '2026

Now extract the `start` strings into an array called `srisetimes`:

In [16]:
#ANSWER:
sriestimes = [xpass['start'] for xpass in passes]
display(sriestimes)

['2026-02-04T09:01:26.281Z',
 '2026-02-04T10:35:16.281Z',
 '2026-02-04T12:13:06.281Z',
 '2026-02-04T13:52:56.281Z',
 '2026-02-04T15:31:26.281Z',
 '2026-02-04T17:07:51.281Z',
 '2026-02-04T18:44:31.281Z',
 '2026-02-04T20:23:56.281Z',
 '2026-02-05T09:48:41.281Z']

These are strings. We convert these to an array of Python `datetime` values called `risetimes`:

In [17]:
risetimes = [datetime.strptime(xpass['start'], "%Y-%m-%dT%H:%M:%S.%fZ") for xpass in passes]
risetimes

[datetime.datetime(2026, 2, 4, 9, 1, 26, 281000),
 datetime.datetime(2026, 2, 4, 10, 35, 16, 281000),
 datetime.datetime(2026, 2, 4, 12, 13, 6, 281000),
 datetime.datetime(2026, 2, 4, 13, 52, 56, 281000),
 datetime.datetime(2026, 2, 4, 15, 31, 26, 281000),
 datetime.datetime(2026, 2, 4, 17, 7, 51, 281000),
 datetime.datetime(2026, 2, 4, 18, 44, 31, 281000),
 datetime.datetime(2026, 2, 4, 20, 23, 56, 281000),
 datetime.datetime(2026, 2, 5, 9, 48, 41, 281000)]

Finally, use `risetime.strftime` to print these in a format that people understand:

```
e.g.
18/10/22 07:05
18/10/22 08:41
18/10/22 10:20
18/10/22 12:00
18/10/22 01:37
18/10/22 03:13
```



In [20]:
#ANSWER:
for xpass in risetimes:
    print(f"{xpass.strftime("%m/%d/%y/ %H:%M")}")

02/04/26/ 09:01
02/04/26/ 10:35
02/04/26/ 12:13
02/04/26/ 13:52
02/04/26/ 15:31
02/04/26/ 17:07
02/04/26/ 18:44
02/04/26/ 20:23
02/05/26/ 09:48


Finally, here is an endpoint that tells us who is on board:

In [21]:
response = requests.get("http://api.open-notify.org/astros.json")

Referring to the methods used above, extract the number of astronauts and their names:

In [ ]:
#ANSWER:


## HOMEWORK


1. Write a simple handler for the response status code (refer to lab resources slide for HTTP response codes). As this Jupyter Notebook is an interactive device, the handler does not need to manage subsequent code execution (i.e. by branching or aborting execution), although it should return something that could be used to do so if deployed in a Python program.

In [22]:
#ANSWER:
def handleResponse(response, verbose = False):
    if response.status_code == 200:
        return False, response.status_code
    else:
        return True, response.status_code
        
  # if Status Code is 200 return false, and status code
  # Otherwise Return True and Status Code

2. Test your response handler on some correct and incorrect API calls.

In [23]:
response = requests.get("http://api.open-notify.org/astros.json")
if handleResponse(response)[0]:
    print('API call failed. Resolve issue before continuing!')

response = requests.get("http://api.open-notify.org/iss-now.json")
handleResponse(response, True)[0]

False

>

>

>



---



---



> > > > > > > > > © 2025 Institute of Data


---



---



